Entity

In [3]:
from dataclasses import dataclass
from pathlib import Path
import os
 
@dataclass(frozen=True)
class DataPreprocessingConfig:
    root_dir: Path
    train_dir: Path
    test_dir: Path
    input_size: int
    resize_size: int
    randaugment_num_ops: int
    randaugment_magnitude: int
    random_erasing_p: float
 

In [4]:
os.getcwd()

'd:\\Personal_projects\\Pyhton_proj\\AI-Food-Recognition-Nutrition-Assistant\\research'

In [5]:
os.chdir("..")

In [6]:
%pwd

'd:\\Personal_projects\\Pyhton_proj\\AI-Food-Recognition-Nutrition-Assistant'

Config Manager

In [7]:
from AI_Food_Recognition_Nutrition_Assistant.constants import *
from AI_Food_Recognition_Nutrition_Assistant.utils.common import read_yaml,create_directories

In [9]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_preprocessing_config(self) -> DataPreprocessingConfig:
        config = self.config.data_preprocessing
        p = self.params
        ensure_dir(Path(config.root_dir))
        data_preprocessing_config = DataPreprocessingConfig(
            root_dir=Path(config.root_dir),
            train_dir=Path(config.train_dir),
            test_dir=Path(config.test_dir),
            input_size=p.model.input_size,
            resize_size=p.model.resize_size,
            randaugment_num_ops=p.augmentation.randaugment_num_ops,
            randaugment_magnitude=p.augmentation.randaugment_magnitude,
            random_erasing_p=p.augmentation.random_erasing_p,
        )

        return data_preprocessing_config

Data Preprocessing Component

In [ ]:
from torchvision import transforms
from torchvision.transforms import RandAugment
from AI_Food_Recognition_Nutrition_Assistant import logger

In [ ]:
class DataPreprocessing:
    def __init__(self, config: DataPreprocessingConfig, batch_size: int, num_workers: int):
        self.config = config
        self.batch_size = batch_size
        self.num_workers = num_workers
 
    def get_train_transform(self) -> transforms.Compose:
        return transforms.Compose([
            transforms.RandomResizedCrop(self.config.input_size),
            transforms.RandomHorizontalFlip(),
            RandAugment(
                num_ops=self.config.randaugment_num_ops,
                magnitude=self.config.randaugment_magnitude
            ),
            transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
            transforms.RandomErasing(
                p=self.config.random_erasing_p,
                scale=(0.02, 0.2),
                ratio=(0.3, 3.3)
            ),
        ])
 
    def get_test_transform(self) -> transforms.Compose:
        return transforms.Compose([
            transforms.Resize(self.config.resize_size),
            transforms.CenterCrop(self.config.input_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])
 
    def get_dataloaders(self):
        root = self.config.root_dir
        train_ds = Food101(
            root=root, split="train",
            transform=self.get_train_transform(), download=False
        )
        test_ds = Food101(
            root=root, split="test",
            transform=self.get_test_transform(), download=False
        )
        train_loader = DataLoader(
            train_ds, batch_size=self.batch_size,
            shuffle=True, num_workers=self.num_workers,
            pin_memory=True, prefetch_factor=4
        )
        test_loader = DataLoader(
            test_ds, batch_size=self.batch_size,
            shuffle=False, num_workers=self.num_workers,
            pin_memory=True, prefetch_factor=4
        )
        logger.info(f"DataLoaders ready. Train: {len(train_ds)}, Test: {len(test_ds)}")
        return train_loader, test_loader, train_ds.classes

Pipeline

In [19]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zipfile()
except Exception as e:
    raise e

[2026-04-16 15:58:20,420: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='utf-8'> loaded successfully]
[2026-04-16 15:58:20,450: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='utf-8'> loaded successfully]
[2026-04-16 15:58:20,465: INFO: common: created directory at: artifacts]
[2026-04-16 15:58:20,467: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-16 16:05:06,190: INFO: 3471712837: artifacts/data_ingestion/data.zip downloaded successfully with info Content-Type: application/zip
X-GUploader-UploadID: AMNfjG3Kf7BQ03uO9ZRw-sMjsgvAPv3rZCz19Ud0MBMUVOUzvGaKeQl2Dv-KbhgrA5XEGlDd
Expires: Thu, 16 Apr 2026 13:58:21 GMT
Date: Thu, 16 Apr 2026 13:58:21 GMT
Cache-Control: private, max-age=0
Last-Modified: Thu, 21 Nov 2019 01:36:33 GMT
ETag: "4c697f34e0f1b5db4b95cb3793c02592"
x-goog-generation: 1574300193295556
x-goog-metageneration: 1
x-goog-stored-content-encoding: identity
x-goog-stored-content-l